In [ ]:
%%writefile naive.cu
#include <iostream>
#include <random>
#include <cuda_runtime.h>
#ifndef uint
#define uint unsigned int
#endif


#define CUDA_CHECK(err)                                                   \
    do {                                                                  \
        cudaError_t e = (err);                                            \
        if (e != cudaSuccess) {                                           \
            std::cerr << "CUDA error " << __FILE__ << ':' << __LINE__      \
                      << " : " << cudaGetErrorString(e) << std::endl;      \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                 \
    } while (0)

// ---------------------------------------------------------------------
// Naïve kernel (row‑major, one thread per output element)
// ---------------------------------------------------------------------
__global__ void naive(int M, int N, int K,
                            float alpha,
                            const float* A,
                            const float* B,
                            float beta,
                            float* C)
{
    const uint x = blockIdx.x * blockDim.x + threadIdx.x; // row
    const uint y = blockIdx.y * blockDim.y + threadIdx.y; // col

    if (x < M && y < N) {
        float acc = 0.0f;
        for (int i = 0; i < K; ++i){
            acc += A[x * K + i] * B[i * N + y];
        }

        C[x * N + y] = alpha * acc + beta * C[x * N + y];
    }
}

// ---------------------------------------------------------------------
// Fill a buffer with deterministic pseudo‑random numbers [0,1)
// ---------------------------------------------------------------------
static void random_fill(float* p, size_t n)
{
    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    for (size_t i = 0; i < n; ++i) p[i] = dist(rng);
}

// ---------------------------------------------------------------------
// GFLOPS = 2·M·N·K / (seconds·1e9)
// ---------------------------------------------------------------------
static double gflops(int M, int N, int K, double sec)
{
    return 2.0 * M * N * K / (sec * 1e9);
}

// --------------------------------------------------------------------
// Main driver
// ---------------------------------------------------------------------
int main()
{
    std::cout << "Running Naive Implementation" << std::endl;

    const int maxSize = 4096;
    const int sizes[] = {128, 256, 512, 1024, 2048, 4096};
    const int nSizes   = sizeof(sizes) / sizeof(sizes[0]);

    // --------------------------------------------------------------
    // Allocate host buffers for the *maximum* matrix size
    // --------------------------------------------------------------
    size_t maxElems = static_cast<size_t>(maxSize) * maxSize;
    float *hA = (float*)malloc(maxElems * sizeof(float));
    float *hB = (float*)malloc(maxElems * sizeof(float));
    float *hC = (float*)malloc(maxElems * sizeof(float));
    if (!hA || !hB || !hC) {
        std::cerr << "Host allocation failed\n";
        return EXIT_FAILURE;
    }
    random_fill(hA, maxElems);
    random_fill(hB, maxElems);
    random_fill(hC, maxElems);

    // --------------------------------------------------------------
    // Allocate device buffers (same max size) and copy once
    // --------------------------------------------------------------
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dB, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dC, maxElems * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(dA, hA, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dC, hC, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));

    const dim3 block(16, 16);
    const float alpha = 0.5f;
    const float beta  = 3.0f;
    const int repeat = 50;               // timed launches per size

    // --------------------------------------------------------------
    // Loop over the six matrix sizes
    // --------------------------------------------------------------
    for (int i = 0; i < nSizes; ++i) {
        int M = sizes[i];
        int N = sizes[i];
        int K = sizes[i];

        dim3 grid( (M + block.x - 1) / block.x,
                   (N + block.y - 1) / block.y );

        // Warm‑up (removes first‑run overhead)
        naive<<<grid, block>>>(M, N, K, alpha, dA, dB, beta, dC);
        CUDA_CHECK(cudaDeviceSynchronize());

        // Timing with CUDA events
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaEventRecord(start));
        for (int r = 0; r < repeat; ++r)
            naive<<<grid, block>>>(M, N, K, alpha, dA, dB, beta, dC);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        ms /= repeat;                     // average per launch (ms)

        double sec = ms * 1e-3;           // seconds for printing
        double perf = gflops(M, N, K, sec);

        // ----- EXACT output format -----
        printf("dimensions(m=n=k) %d, alpha: %.1f, beta: %.1f\n",
               M, alpha, beta);
        printf("Average elapsed time: (%.6f) s, performance: (%8.1f) GFLOPS. size: (%d).\n",
               sec, perf, M);
        // ------------------------------

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }

    // --------------------------------------------------------------
    // Clean‑up
    // --------------------------------------------------------------
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA);
    free(hB);
    free(hC);

    return 0;
}

Writing naive.cu


In [4]:
!nvcc -arch=sm_75 -lcublas naive.cu -o naive.exe
!./naive.exe


Running Naive Implementation
dimensions(m=n=k) 128, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000125) s, performance: (    33.6) GFLOPS. size: (128).
dimensions(m=n=k) 256, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000844) s, performance: (    39.8) GFLOPS. size: (256).
dimensions(m=n=k) 512, alpha: 0.5, beta: 3.0
Average elapsed time: (0.004996) s, performance: (    53.7) GFLOPS. size: (512).
dimensions(m=n=k) 1024, alpha: 0.5, beta: 3.0
Average elapsed time: (0.018215) s, performance: (   117.9) GFLOPS. size: (1024).
dimensions(m=n=k) 2048, alpha: 0.5, beta: 3.0
Average elapsed time: (0.146795) s, performance: (   117.0) GFLOPS. size: (2048).
dimensions(m=n=k) 4096, alpha: 0.5, beta: 3.0
Average elapsed time: (1.234509) s, performance: (   111.3) GFLOPS. size: (4096).


In [5]:
%%writefile global_mem_coalesce.cu

//==========================================================================
// global_mem_coalesce.cu – GEMM with coalesced global loads
//==========================================================================
#ifndef uint
#define uint unsigned int
#endif


#include <iostream>
#include <random>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <cstdint>          // for uint32_t

// ---------------------------------------------------------------------
// Simple error‑checking macro
// ---------------------------------------------------------------------
#define CUDA_CHECK(err)                                                   \
    do {                                                                  \
        cudaError_t e = (err);                                            \
        if (e != cudaSuccess) {                                           \
            std::cerr << "CUDA error " << __FILE__ << ':' << __LINE__      \
                      << " : " << cudaGetErrorString(e) << std::endl;      \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                 \
    } while (0)

// ---------------------------------------------------------------------
// Integer ceiling division (used for grid sizing)
// ---------------------------------------------------------------------
#define CEIL_DIV(x, y)   ( ((x) + (y) - 1) / (y) )

// ---------------------------------------------------------------------
// GEMM kernel – global‑memory coalescing
// ---------------------------------------------------------------------
// BLOCKSIZE is the tile dimension (e.g. 16).  The block is 1‑D with
// BLOCKSIZE*BLOCKSIZE threads.  Each thread computes one C element.
template <uint32_t BLOCKSIZE>
__global__ void global_mem_coalesce(int M, int N, int K,
                                   float alpha,
                                   const float* A,
                                   const float* B,
                                   float beta,
                                   float* C)
{
    // 1‑D thread index → (row, col) inside the tile
    const int cRow = blockIdx.x * BLOCKSIZE + (threadIdx.x / BLOCKSIZE);
    const int cCol = blockIdx.y * BLOCKSIZE + (threadIdx.x % BLOCKSIZE);

    if (cRow < M && cCol < N) {
        float acc = 0.0f;
        for (int i = 0; i < K; ++i) {
            // Row‑major A, column‑major B (same as naïve)
            acc += A[cRow * K + i] * B[i * N + cCol];
        }
        C[cRow * N + cCol] = alpha * acc + beta * C[cRow * N + cCol];
    }
}

void run_global_mem_coalesce(int M, int N, int K,
                             float alpha, float* dA, float* dB,
                             float beta,  float* dC)
{
    // Choose a tile size that matches the kernel template.
    // 32 → 32×32 = 1024 threads per block (max for modern GPUs).
    constexpr uint32_t TILE = 32;

    dim3 grid( CEIL_DIV(M, TILE), CEIL_DIV(N, TILE) );
    dim3 block( TILE * TILE );               // 1‑D block

    // Instantiate the templated kernel with BLOCKSIZE = TILE
    global_mem_coalesce<TILE><<<grid, block>>>(M, N, K,
                                              alpha, dA, dB,
                                              beta,  dC);
}

// ---------------------------------------------------------------------
// Fill a buffer with deterministic pseudo‑random numbers [0,1)
// ---------------------------------------------------------------------
static void random_fill(float* p, size_t n)
{
    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    for (size_t i = 0; i < n; ++i) p[i] = dist(rng);
}

// ---------------------------------------------------------------------
// GFLOPS = 2·M·N·K / (seconds·1e9)
// ---------------------------------------------------------------------
static double gflops(int M, int N, int K, double sec)
{
    return 2.0 * M * N * K / (sec * 1e9);
}

// ---------------------------------------------------------------------
// Main driver
// ---------------------------------------------------------------------
int main()
{
    std::cout << "Running Global‑Memory‑Coalesce Implementation" << std::endl;

    const int maxSize = 4096;
    const int sizes[] = {128, 256, 512, 1024, 2048, 4096};
    const int nSizes   = sizeof(sizes) / sizeof(sizes[0]);

    // --------------------------------------------------------------
    // Allocate host buffers for the *maximum* matrix size
    // --------------------------------------------------------------
    size_t maxElems = static_cast<size_t>(maxSize) * maxSize;
    float *hA = (float*)malloc(maxElems * sizeof(float));
    float *hB = (float*)malloc(maxElems * sizeof(float));
    float *hC = (float*)malloc(maxElems * sizeof(float));
    if (!hA || !hB || !hC) {
        std::cerr << "Host allocation failed\n";
        return EXIT_FAILURE;
    }

    random_fill(hA, maxElems);
    random_fill(hB, maxElems);
    random_fill(hC, maxElems);

    // --------------------------------------------------------------
    // Allocate device buffers (same max size) and copy once
    // --------------------------------------------------------------
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dB, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dC, maxElems * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(dA, hA, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dC, hC, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));

    // --------------------------------------------------------------
    // Benchmark parameters
    // --------------------------------------------------------------
    const float alpha = 0.5f;
    const float beta  = 3.0f;
    const int   repeat = 50;               // timed launches per size

    // --------------------------------------------------------------
    // Loop over the six matrix sizes
    // --------------------------------------------------------------
    for (int i = 0; i < nSizes; ++i) {
        int M = sizes[i];
        int N = sizes[i];
        int K = sizes[i];

        // Warm‑up launch (removes first‑run overhead)
        run_global_mem_coalesce(M, N, K, alpha, dA, dB, beta, dC);
        CUDA_CHECK(cudaDeviceSynchronize());

        // ----------------------------------------------------------
        // Timing with CUDA events
        // ----------------------------------------------------------
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaEventRecord(start));
        for (int r = 0; r < repeat; ++r) {
            run_global_mem_coalesce(M, N, K, alpha, dA, dB, beta, dC);
        }
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        ms /= repeat;                     // average per launch (ms)

        double sec = ms * 1e-3;           // seconds for printing
        double perf = gflops(M, N, K, sec);

        // ----- EXACT output format -----
        printf("dimensions(m=n=k) %d, alpha: %.1f, beta: %.1f\n",
               M, alpha, beta);
        printf("Average elapsed time: (%.6f) s, performance: (%8.1f) GFLOPS. size: (%d).\n",
               sec, perf, M);
        // ------------------------------

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }

    // --------------------------------------------------------------
    // Clean‑up
    // --------------------------------------------------------------
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA);
    free(hB);
    free(hC);

    return 0;
}

Writing global_mem_coalesce.cu


In [7]:
!nvcc -arch=sm_75 -lcublas global_mem_coalesce.cu -o global_mem_coalesce.exe
!./global_mem_coalesce.exe

Running Global‑Memory‑Coalesce Implementation
dimensions(m=n=k) 128, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000035) s, performance: (   119.3) GFLOPS. size: (128).
dimensions(m=n=k) 256, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000131) s, performance: (   255.8) GFLOPS. size: (256).
dimensions(m=n=k) 512, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000885) s, performance: (   303.4) GFLOPS. size: (512).
dimensions(m=n=k) 1024, alpha: 0.5, beta: 3.0
Average elapsed time: (0.004484) s, performance: (   479.0) GFLOPS. size: (1024).
dimensions(m=n=k) 2048, alpha: 0.5, beta: 3.0
Average elapsed time: (0.030695) s, performance: (   559.7) GFLOPS. size: (2048).
dimensions(m=n=k) 4096, alpha: 0.5, beta: 3.0
Average elapsed time: (0.252106) s, performance: (   545.2) GFLOPS. size: (4096).


In [10]:
%%writefile shared_mem_block.cu
//==========================================================================
// shared_mem_block.cu – GEMM with shared‑memory blocking
//==========================================================================
#ifndef uint
#define uint unsigned int
#endif


#include <iostream>
#include <random>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <cstdint>          // for uint32_t

// ---------------------------------------------------------------------
// Simple error‑checking macro
// ---------------------------------------------------------------------
#define CUDA_CHECK(err)                                                   \
    do {                                                                  \
        cudaError_t e = (err);                                            \
        if (e != cudaSuccess) {                                           \
            std::cerr << "CUDA error " << __FILE__ << ':' << __LINE__      \
                      << " : " << cudaGetErrorString(e) << std::endl;      \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                 \
    } while (0)

// ---------------------------------------------------------------------
// Integer ceiling division (used for grid sizing)
// ---------------------------------------------------------------------
#define CEIL_DIV(x, y)   ( ((x) + (y) - 1) / (y) )

// ---------------------------------------------------------------------
// GEMM kernel – shared‑memory blocking
// ---------------------------------------------------------------------
// BLOCKSIZE is the tile dimension (e.g. 32).  The block is 1‑D with
// BLOCKSIZE*BLOCKSIZE threads.  Each thread computes one element of C.
template <uint32_t BLOCKSIZE>
__global__ void shared_mem_block(int M, int N, int K,
                                 float alpha,
                                 const float* A,
                                 const float* B,
                                 float beta,
                                 float* C)
{
    const uint cRow = blockIdx.x;                     // tile row
    const uint cCol = blockIdx.y;                     // tile col

    __shared__ float As[BLOCKSIZE * BLOCKSIZE];
    __shared__ float Bs[BLOCKSIZE * BLOCKSIZE];

    const uint threadCol = threadIdx.x % BLOCKSIZE; // coordinates of thread inside the tile that the block is working on
    const uint threadRow = threadIdx.x / BLOCKSIZE;

    // advance pointers to the first element of this tile
    A += cRow * BLOCKSIZE * K;                       // row = cRow, col = 0
    B += cCol * BLOCKSIZE;                           // row = 0,   col = cCol
    C += cRow * BLOCKSIZE * N + cCol * BLOCKSIZE;    // row = cRow, col = cCol

    float tmp = 0.0f;

    for (int bkIdx = 0; bkIdx < K; bkIdx += BLOCKSIZE) {
        // load one tile of A and B into shared memory (coalesced)
        As[threadRow * BLOCKSIZE + threadCol] = A[threadRow * K + threadCol];
        Bs[threadRow * BLOCKSIZE + threadCol] = B[threadRow * N + threadCol];
        __syncthreads();

        // compute partial dot‑product using the shared tiles
        for (int dotIdx = 0; dotIdx < BLOCKSIZE; ++dotIdx) {
            tmp += As[threadRow * BLOCKSIZE + dotIdx] *
                   Bs[dotIdx * BLOCKSIZE + threadCol];
        }
        __syncthreads();

        // move to the next K‑chunk
        A += BLOCKSIZE;
        B += BLOCKSIZE * N;
    }

    C[threadRow * N + threadCol] = alpha * tmp + beta * C[threadRow * N + threadCol];
}

// ---------------------------------------------------------------------
// Helper that mirrors the “run_global_mem_coalesce” style
// ---------------------------------------------------------------------
void run_shared_mem_block(int M, int N, int K,
                         float alpha, float* dA, float* dB,
                         float beta,  float* dC)
{
    constexpr uint32_t TILE = 32;                     // 32×32 = 1024 threads
    dim3 grid( CEIL_DIV(M, TILE), CEIL_DIV(N, TILE) );
    dim3 block( TILE * TILE );                        // 1‑D block

    // carve out as much L1 as possible for shared memory (optional)
    cudaFuncSetAttribute(
        shared_mem_block<TILE>,
        cudaFuncAttributePreferredSharedMemoryCarveout,
        cudaSharedmemCarveoutMaxShared);

    shared_mem_block<TILE><<<grid, block>>>(M, N, K,
                                           alpha, dA, dB,
                                           beta,  dC);
}

// ---------------------------------------------------------------------
// Fill a buffer with deterministic pseudo‑random numbers [0,1)
// ---------------------------------------------------------------------
static void random_fill(float* p, size_t n)
{
    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    for (size_t i = 0; i < n; ++i) p[i] = dist(rng);
}

// ---------------------------------------------------------------------
// GFLOPS = 2·M·N·K / (seconds·1e9)
// ---------------------------------------------------------------------
static double gflops(int M, int N, int K, double sec)
{
    return 2.0 * M * N * K / (sec * 1e9);
}

// ---------------------------------------------------------------------
// Main driver
// ---------------------------------------------------------------------
int main()
{
    std::cout << "Running Shared‑Memory‑Block Implementation" << std::endl;

    const int maxSize = 4096;
    const int sizes[] = {128, 256, 512, 1024, 2048, 4096};
    const int nSizes   = sizeof(sizes) / sizeof(sizes[0]);

    // --------------------------------------------------------------
    // Allocate host buffers for the *maximum* matrix size
    // --------------------------------------------------------------
    size_t maxElems = static_cast<size_t>(maxSize) * maxSize;
    float *hA = (float*)malloc(maxElems * sizeof(float));
    float *hB = (float*)malloc(maxElems * sizeof(float));
    float *hC = (float*)malloc(maxElems * sizeof(float));
    if (!hA || !hB || !hC) {
        std::cerr << "Host allocation failed\n";
        return EXIT_FAILURE;
    }

    random_fill(hA, maxElems);
    random_fill(hB, maxElems);
    random_fill(hC, maxElems);

    // --------------------------------------------------------------
    // Allocate device buffers (same max size) and copy once
    // --------------------------------------------------------------
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dB, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dC, maxElems * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(dA, hA, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dC, hC, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));

    // --------------------------------------------------------------
    // Benchmark parameters
    // --------------------------------------------------------------
    const float alpha = 0.5f;
    const float beta  = 3.0f;
    const int   repeat = 50;               // timed launches per size

    // --------------------------------------------------------------
    // Loop over the six matrix sizes
    // --------------------------------------------------------------
    for (int i = 0; i < nSizes; ++i) {
        int M = sizes[i];
        int N = sizes[i];
        int K = sizes[i];

        // Warm‑up launch (removes first‑run overhead)
        run_shared_mem_block(M, N, K, alpha, dA, dB, beta, dC);
        CUDA_CHECK(cudaDeviceSynchronize());

        // ----------------------------------------------------------
        // Timing with CUDA events
        // ----------------------------------------------------------
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaEventRecord(start));
        for (int r = 0; r < repeat; ++r) {
            run_shared_mem_block(M, N, K, alpha, dA, dB, beta, dC);
        }
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        ms /= repeat;                     // average per launch (ms)

        double sec = ms * 1e-3;           // seconds for printing
        double perf = gflops(M, N, K, sec);

        // ----- EXACT output format -----
        printf("dimensions(m=n=k) %d, alpha: %.1f, beta: %.1f\n",
               M, alpha, beta);
        printf("Average elapsed time: (%.6f) s, performance: (%8.1f) GFLOPS. size: (%d).\n",
               sec, perf, M);
        // ------------------------------

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }

    // --------------------------------------------------------------
    // Clean‑up
    // --------------------------------------------------------------
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA);
    free(hB);
    free(hC);

    return 0;
}

Overwriting shared_mem_block.cu


In [11]:
!nvcc -arch=sm_75 -lcublas shared_mem_block.cu -o shared_mem_block.exe
!./shared_mem_block.exe

Running Shared‑Memory‑Block Implementation
dimensions(m=n=k) 128, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000025) s, performance: (   165.1) GFLOPS. size: (128).
dimensions(m=n=k) 256, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000106) s, performance: (   317.5) GFLOPS. size: (256).
dimensions(m=n=k) 512, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000701) s, performance: (   382.9) GFLOPS. size: (512).
dimensions(m=n=k) 1024, alpha: 0.5, beta: 3.0
Average elapsed time: (0.003910) s, performance: (   549.3) GFLOPS. size: (1024).
dimensions(m=n=k) 2048, alpha: 0.5, beta: 3.0
Average elapsed time: (0.019594) s, performance: (   876.8) GFLOPS. size: (2048).
dimensions(m=n=k) 4096, alpha: 0.5, beta: 3.0
Average elapsed time: (0.158159) s, performance: (   869.0) GFLOPS. size: (4096).


In [13]:
%%writefile 1D_block_tiling.cu
//==========================================================================
// 1d_block_tiling.cu – GEMM with 1‑D block tiling (register‑blocked)
//==========================================================================
#ifndef uint
#define uint unsigned int
#endif


#include <iostream>
#include <random>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <cstdint>          // for uint32_t
#include <cassert>          // for assert

// ---------------------------------------------------------------------
// Simple error‑checking macro
// ---------------------------------------------------------------------
#define CUDA_CHECK(err)                                                   \
    do {                                                                  \
        cudaError_t e = (err);                                            \
        if (e != cudaSuccess) {                                           \
            std::cerr << "CUDA error " << __FILE__ << ':' << __LINE__      \
                      << " : " << cudaGetErrorString(e) << std::endl;      \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                 \
    } while (0)

// ---------------------------------------------------------------------
// Integer ceiling division (used for grid sizing)
// ---------------------------------------------------------------------
#define CEIL_DIV(x, y)   ( ((x) + (y) - 1) / (y) )

// ---------------------------------------------------------------------
// GEMM kernel – 1‑D block tiling
// ---------------------------------------------------------------------
template <int BM, int BN, int BK, int TM>
__global__ void sgemm1DBlocktiling(int M, int N, int K,
                                   float alpha,
                                   const float *A,
                                   const float *B,
                                   float beta,
                                   float *C)
{
    // Tile indices (each block works on a BM×BN tile of C)
    const unsigned int cRow = blockIdx.y;   // tile‑row
    const unsigned int cCol = blockIdx.x;   // tile‑col

    // Thread coordinates inside the tile
    const int threadCol = threadIdx.x % BN;   // 0 … BN‑1
    const int threadRow = threadIdx.x / BN;   // 0 … BM‑1

    // Shared‑memory buffers for the current A‑ and B‑tiles
    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    // Move the global pointers to the first element of this tile
    A += cRow * BM * K;                     // start of the tile’s rows in A
    B += cCol * BN;                         // start of the tile’s columns in B
    C += cRow * BM * N + cCol * BN;          // top‑left element of the output tile

    // Sanity checks – they fire if the launch configuration is wrong
    assert(BM * BK == blockDim.x);
    assert(BN * BK == blockDim.x);

    // Ind the shared‑memory tiles (warp‑level coalescing)
    const unsigned int innerColA = threadIdx.x % BK;
    const unsigned int innerRowA = threadIdx.x / BK;
    const unsigned int innerColB = threadIdx.x % BN;
    const unsigned int innerRowB = threadIdx.x / BN;

    // Register‑level accumulator (each thread produces TM rows)
    float threadResults[TM] = {0.0f};

    // -----------------------------------------------------------------
    // Loop over the K dimension in BK‑wide chunks
    // -----------------------------------------------------------------
    for (unsigned int bkIdx = 0; bkIdx < K; bkIdx += BK)
    {
        // Load one BM×BK tile of A and one BK×BN tile of B into shared memory
        As[innerRowA * BK + innerColA] = A[innerRowA * K + innerColA];
        Bs[innerRowB * BN + innerColB] = B[innerRowB * N + innerColB];
        __syncthreads();
        // __syncth Advance the global pointers to the next K-chunk
        A += BK;            // move right by BK columns in A
        B += BK * N;        // move down by BK rows in B

        // Compute the partial dot‑product for this chunk
        for (unsigned int dotIdx = 0; dotIdx < BK; ++dotIdx)
        {
            float tmpB = Bs[dotIdx * BN + threadCol];
            for (unsigned int resIdx = 0; resIdx < TM; ++resIdx)
            {
                threadResults[resIdx] +=
                    As[(threadRow * TM + resIdx) * BK + dotIdx] * tmpB;
            }
        }
        __syncthreads();
    }

    // -----------------------------------------------------------------
    // Write the final results back to global memory
    // -----------------------------------------------------------------
    for (unsigned int resIdx = 0; resIdx < TM; ++resIdx)
    {
        C[(threadRow * TM + resIdx) * N + threadCol] =
            alpha * threadResults[resIdx] +
            beta  * C[(threadRow * TM + resIdx) * N + threadCol];
    }
}

void run1DBlocktiling(int M, int N, int K,
                      float alpha,
                      float *A, float *B,
                      float beta, float *C)
{
    constexpr unsigned int BM = 64;
    constexpr unsigned int BN = 64;
    constexpr unsigned int BK = 8;
    constexpr unsigned int TM = 8;

    dim3 gridDim( CEIL_DIV(N, BN), CEIL_DIV(M, BM) );
    dim3 blockDim( (BM * BN) / TM );   // e.g. (64*64)/8 = 512 threads

    sgemm1DBlocktiling<BM, BN, BK, TM>
        <<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}

// ---------------------------------------------------------------------
// Fill a buffer with deterministic pseudo‑random numbers [0,1)
// ---------------------------------------------------------------------
static void random_fill(float* p, size_t n)
{
    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    for (size_t i = 0; i < n; ++i) p[i] = dist(rng);
}

// ---------------------------------------------------------------------
// GFLOPS = 2·M·N·K / (seconds·1e9)
// ---------------------------------------------------------------------
static double gflops(int M, int N, int K, double sec)
{
    return 2.0 * M * N * K / (sec * 1e9);
}

// ---------------------------------------------------------------------
// Main driver
// ---------------------------------------------------------------------
int main()
{
    std::cout << "Running 1D‑Block‑Tiling Implementation" << std::endl;

    const int maxSize = 4096;
    const int sizes[] = {128, 256, 512, 1024, 2048, 4096};
    const int nSizes   = sizeof(sizes) / sizeof(sizes[0]);

    // --------------------------------------------------------------
    // Allocate host buffers for the *maximum* matrix size
    // --------------------------------------------------------------
    size_t maxElems = static_cast<size_t>(maxSize) * maxSize;
    float *hA = (float*)malloc(maxElems * sizeof(float));
    float *hB = (float*)malloc(maxElems * sizeof(float));
    float *hC = (float*)malloc(maxElems * sizeof(float));
    if (!hA || !hB || !hC) {
        std::cerr << "Host allocation failed\n";
        return EXIT_FAILURE;
    }

    random_fill(hA, maxElems);
    random_fill(hB, maxElems);
    random_fill(hC, maxElems);

    // --------------------------------------------------------------
    // Allocate device buffers (same max size) and copy once
    // --------------------------------------------------------------
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dB, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dC, maxElems * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(dA, hA, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dC, hC, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));

    // --------------------------------------------------------------
    // Benchmark parameters
    // --------------------------------------------------------------
    const float alpha = 0.5f;
    const float beta  = 3.0f;
    const int   repeat = 50;               // timed launches per size

    // --------------------------------------------------------------
    // Loop over the six matrix sizes
    // --------------------------------------------------------------
    for (int i = 0; i < nSizes; ++i) {
        int M = sizes[i];
        int N = sizes[i];
        int K = sizes[i];

        // Warm‑up launch (removes first‑run overhead)
        run1DBlocktiling(M, N, K, alpha, dA, dB, beta, dC);
        CUDA_CHECK(cudaDeviceSynchronize());

        // ----------------------------------------------------------
        // Timing with CUDA events
        // ----------------------------------------------------------
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaEventRecord(start));
        for (int r = 0; r < repeat; ++r) {
            run1DBlocktiling(M, N, K, alpha, dA, dB, beta, dC);
        }
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        ms /= repeat;                     // average per launch (ms)

        double sec = ms * 1e-3;           // seconds for printing
        double perf = gflops(M, N, K, sec);

        // ----- EXACT output format -----
        printf("dimensions(m=n=k) %d, alpha: %.1f, beta: %.1f\n",
               M, alpha, beta);
        printf("Average elapsed time: (%.6f) s, performance: (%8.1f) GFLOPS. size: (%d).\n",
               sec, perf, M);
        // ------------------------------

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }

    // --------------------------------------------------------------
    // Clean‑up
    // --------------------------------------------------------------
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA);
    free(hB);
    free(hC);

    return 0;
}

Overwriting 1D_block_tiling.cu


In [14]:
!nvcc -arch=sm_75 -lcublas 1D_block_tiling.cu -o 1D_block_tiling.exe
!./1D_block_tiling.exe

Running 1D‑Block‑Tiling Implementation
dimensions(m=n=k) 128, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000044) s, performance: (    95.4) GFLOPS. size: (128).
dimensions(m=n=k) 256, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000081) s, performance: (   413.7) GFLOPS. size: (256).
dimensions(m=n=k) 512, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000377) s, performance: (   711.7) GFLOPS. size: (512).
dimensions(m=n=k) 1024, alpha: 0.5, beta: 3.0
Average elapsed time: (0.002648) s, performance: (   811.0) GFLOPS. size: (1024).
dimensions(m=n=k) 2048, alpha: 0.5, beta: 3.0
Average elapsed time: (0.009261) s, performance: (  1855.1) GFLOPS. size: (2048).
dimensions(m=n=k) 4096, alpha: 0.5, beta: 3.0
Average elapsed time: (0.074056) s, performance: (  1855.9) GFLOPS. size: (4096).


In [15]:
%%writefile 2D_block_tiling.cu
//==========================================================================
// 2D_block_tiling.cu – GEMM with 2‑D block tiling (register‑blocked)
//==========================================================================
// for Windows adding below block of code
#ifndef uint
#define uint unsigned int
#endif
//


#include <iostream>
#include <random>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <cstdint>          // for uint32_t
#include <cassert>
#include <algorithm>
#include <cstdio>
#include <cstdlib>
#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))       // for assert

// ---------------------------------------------------------------------
// Simple error‑checking macro
// ---------------------------------------------------------------------
#define CUDA_CHECK(err)                                                   \
    do {                                                                  \
        cudaError_t e = (err);                                            \
        if (e != cudaSuccess) {                                           \
            std::cerr << "CUDA error " << __FILE__ << ':' << __LINE__      \
                      << " : " << cudaGetErrorString(e) << std::endl;      \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                 \
    } while (0)

// ---------------------------------------------------------------------
// Kernel – 2‑D block tiling
// ---------------------------------------------------------------------

template <const int BM, const int BN, const int BK, const int TM, const int TN>
__global__ void __launch_bounds__((BM * BN) / (TM * TN), 1)
sgemm2DBlocktiling(int M, int N, int K, float alpha, const float *A,
                   const float *B, float beta, float *C)
{
    // -----------------------------------------------------------------
    // 1) Identify the output tile this block will compute
    // -----------------------------------------------------------------
    const uint cRow = blockIdx.y;                     // tile‑row index
    const uint cCol = blockIdx.x;                     // tile‑col index
    const uint totalResultsBlocktile = BM * BN;        // #output elements per tile

    // A thread produces TM × TN results inside the tile
    const uint numThreadsBlocktile = totalResultsBlocktile / (TM * TN);
    // sanity check – the launch must provide exactly this many threads
    assert(numThreadsBlocktile == blockDim.x);

    // BN/TN threads span a column of the tile
    const int threadCol = threadIdx.x % (BN / TN);
    const int threadRow = threadIdx.x / (BN / TN);

    // -----------------------------------------------------------------
    // 2) Shared‑memory tiles for A and B
    // -----------------------------------------------------------------
    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    // Move the global pointers to the first element of this tile
    A += cRow * BM * K;                     // start of A‑tile rows
    B += cCol * BN;                         // start of B‑tile columns
    C += cRow * BM * N + cCol * BN;         // start of C‑tile

    // -----------------------------------------------------------------
    // 3) Indices used for loading the shared‑memory tiles
    // -----------------------------------------------------------------
    const uint innerRowA = threadIdx.x / BK;
    const uint innerColA = threadIdx.x % BK;
    const uint strideA   = numThreadsBlocktile / BK;   // rows of A loaded per step

    const uint innerRowB = threadIdx.x / BN;
    const uint innerColB = threadIdx.x % BN;
    const uint strideB   = numThreadsBlocktile / BN;   // rows of B loaded per step

    // -----------------------------------------------------------------
    // 4) Thread‑local accumulators (register‑blocked)
    // -----------------------------------------------------------------
    float threadResults[TM * TN] = {0.0f};   // final TM × TN results for this thread
    float regM[TM] = {0.0f};                // registers for a column of A
    float regN[TN] = {0.0f};                // registers for a row    of B

    // -----------------------------------------------------------------
    // 5) Main K‑loop – load tiles, compute, advance pointers
    // -----------------------------------------------------------------
    for (uint bkIdx = 0; bkIdx < K; bkIdx += BK)
    {
        // ---- load a BM×BK tile of A into shared memory -----------------
        for (uint loadOffset = 0; loadOffset < BM; loadOffset += strideA)
        {
            As[(innerRowA + loadOffset) * BK + innerColA] =
                A[(innerRowA + loadOffset) * K + innerColA];
        }

        // ---- load a BK×BN tile of B into shared memory -----------------
        for (uint loadOffset = 0; loadOffset < BK; loadOffset += strideB)
        {
            Bs[(innerRowB + loadOffset) * BN + innerColB] =
                B[(innerRowB + loadOffset) * N + innerColB];
        }

        __syncthreads();

        // ---- advance global pointers to the next K‑chunk ----------------
        A += BK;          // move right by BK columns in A
        B += BK * N;      // move down by BK rows in B

        // ---- compute the partial products for this K‑chunk ------------
        for (uint dotIdx = 0; dotIdx < BK; ++dotIdx)
        {
            // load TM elements of the current column of A into registers
            for (uint i = 0; i < TM; ++i)
                regM[i] = As[(threadRow * TM + i) * BK + dotIdx];

            // load TN elements of the current row of B into registers
            for (uint i = 0; i < TN; ++i)
                regN[i] = Bs[dotIdx * BN + threadCol * TN + i];

            // accumulate TM × TN products
            for (uint resIdxM = 0; resIdxM < TM; ++resIdxM)
            {
                for (uint resIdxN = 0; resIdxN < TN; ++resIdxN)
                {
                    threadResults[resIdxM * TN + resIdxN] +=
                        regM[resIdxM] * regN[resIdxN];
                }
            }
        }

        __syncthreads();   // ensure all threads finished before next tile
    }

    // -----------------------------------------------------------------
    // 6) Write the TM × TN results back to global memory
    // -----------------------------------------------------------------
    for (uint resIdxM = 0; resIdxM < TM; ++resIdxM)
    {
        for (uint resIdxN = 0; resIdxN < TN; ++resIdxN)
        {
            uint globalRow = (threadRow * TM + resIdxM);
            uint globalCol = (threadCol * TN + resIdxN);
            C[globalRow * N + globalCol] =
                alpha * threadResults[resIdxM * TN + resIdxN] +
                beta  * C[globalRow * N + globalCol];
        }
    }
}

// ---------------------------------------------------------------------
// Launcher
// ---------------------------------------------------------------------
void runSgemm2DBlocktiling(int M, int N, int K,
                           float alpha, float *A, float *B,
                           float beta, float *C)
{
    const uint BK = 8;
    const uint TM = 8;
    const uint TN = 8;

    if (M >= 128 && N >= 128)
    {
        const uint BM = 128;
        const uint BN = 128;
        dim3 gridDim( CEIL_DIV(N, BN), CEIL_DIV(M, BM) );
        dim3 blockDim( (BM * BN) / (TM * TN) );   //  (128*128)/(8*8) = 256 threads
        sgemm2DBlocktiling<BM, BN, BK, TM, TN>
            <<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
    }
    else
    {
        // fallback for small problems (still a power‑of‑two tile size)
        const uint BM = 64;
        const uint BN = 64;
        dim3 gridDim( CEIL_DIV(N, BN), CEIL_DIV(M, BM) );
        dim3 blockDim( (BM * BN) / (TM * TN) );   //  (64*64)/(8*8) = 64 threads
        sgemm2DBlocktiling<BM, BN, BK, TM, TN>
            <<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
    }
}

// ---------------------------------------------------------------------
// Fill a buffer with deterministic pseudo‑random numbers [0,1)
// ---------------------------------------------------------------------
static void random_fill(float* p, size_t n)
{
    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    for (size_t i = 0; i < n; ++i) p[i] = dist(rng);
}

// ---------------------------------------------------------------------
// GFLOPS = 2·M·N·K / (seconds·1e9)
// ---------------------------------------------------------------------
static double gflops(int M, int N, int K, double sec)
{
    return 2.0 * M * N * K / (sec * 1e9);
}

// ---------------------------------------------------------------------
// Main driver
// ---------------------------------------------------------------------
int main()
{
    std::cout << "Running 2D-Block-Tiling Implementation" << std::endl;

    const int maxSize = 4096;
    const int sizes[] = {128, 256, 512, 1024, 2048, 4096};
    const int nSizes   = sizeof(sizes) / sizeof(sizes[0]);

    // --------------------------------------------------------------
    // Allocate host buffers for the *maximum* matrix size
    // --------------------------------------------------------------
    size_t maxElems = static_cast<size_t>(maxSize) * maxSize;
    float *hA = (float*)malloc(maxElems * sizeof(float));
    float *hB = (float*)malloc(maxElems * sizeof(float));
    float *hC = (float*)malloc(maxElems * sizeof(float));
    if (!hA || !hB || !hC) {
        std::cerr << "Host allocation failed\n";
        return EXIT_FAILURE;
    }

    random_fill(hA, maxElems);
    random_fill(hB, maxElems);
    random_fill(hC, maxElems);

    // --------------------------------------------------------------
    // Allocate device buffers (same max size) and copy once
    // --------------------------------------------------------------
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dB, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dC, maxElems * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(dA, hA, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dC, hC, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));

    // --------------------------------------------------------------
    // Benchmark parameters
    // --------------------------------------------------------------
    const float alpha = 0.5f;
    const float beta  = 3.0f;
    const int   repeat = 50;               // timed launches per size

    // --------------------------------------------------------------
    // Loop over the six matrix sizes
    // --------------------------------------------------------------
    for (int i = 0; i < nSizes; ++i) {
        int M = sizes[i];
        int N = sizes[i];
        int K = sizes[i];

        // Warm‑up launch (removes first‑run overhead)
        runSgemm2DBlocktiling(M, N, K, alpha, dA, dB, beta, dC);
        CUDA_CHECK(cudaDeviceSynchronize());

        // ----------------------------------------------------------
        // Timing with CUDA events
        // ----------------------------------------------------------
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaEventRecord(start));
        for (int r = 0; r < repeat; ++r) {
            runSgemm2DBlocktiling(M, N, K, alpha, dA, dB, beta, dC);
        }
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        ms /= repeat;                     // average per launch (ms)

        double sec = ms * 1e-3;           // seconds for printing
        double perf = gflops(M, N, K, sec);

        // ----- EXACT output format -----
        printf("dimensions(m=n=k) %d, alpha: %.1f, beta: %.1f\n",
               M, alpha, beta);
        printf("Average elapsed time: (%.6f) s, performance: (%8.1f) GFLOPS. size: (%d).\n",
               sec, perf, M);
        // ------------------------------

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }

    // --------------------------------------------------------------
    // Clean‑up
    // --------------------------------------------------------------
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA);
    free(hB);
    free(hC);

    return 0;
}

Writing 2D_block_tiling.cu


In [16]:
!nvcc -arch=sm_75 -lcublas 2D_block_tiling.cu -o 2D_block_tiling.exe
!./2D_block_tiling.exe

Running 2D-Block-Tiling Implementation
dimensions(m=n=k) 128, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000115) s, performance: (    36.4) GFLOPS. size: (128).
dimensions(m=n=k) 256, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000196) s, performance: (   171.0) GFLOPS. size: (256).
dimensions(m=n=k) 512, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000360) s, performance: (   746.4) GFLOPS. size: (512).
dimensions(m=n=k) 1024, alpha: 0.5, beta: 3.0
Average elapsed time: (0.001582) s, performance: (  1357.5) GFLOPS. size: (1024).
dimensions(m=n=k) 2048, alpha: 0.5, beta: 3.0
Average elapsed time: (0.005370) s, performance: (  3199.4) GFLOPS. size: (2048).
dimensions(m=n=k) 4096, alpha: 0.5, beta: 3.0
Average elapsed time: (0.040597) s, performance: (  3385.5) GFLOPS. size: (4096).


In [17]:
%%writefile vectorize.cu
//==========================================================================
//  vectorize.cu – GEMM with 2‑D block tiling and float4 vectorisation
//==========================================================================
#ifndef uint
#define uint unsigned int
#endif


#include <iostream>
#include <random>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <cstdint>          // for uint32_t
#include <cassert>
#include <cstdio>
#include <cstdlib>

// ---------------------------------------------------------------------
// Simple error‑checking macro
// ---------------------------------------------------------------------
#define CUDA_CHECK(err)                                                   \
    do {                                                                  \
        cudaError_t e = (err);                                            \
        if (e != cudaSuccess) {                                           \
            std::cerr << "CUDA error " << __FILE__ << ':' << __LINE__      \
                      << " : " << cudaGetErrorString(e) << std::endl;      \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                 \
    } while (0)

#define CUBLAS_CHECK(err)                                                 \
    do {                                                                  \
        cublasStatus_t s = (err);                                        \
        if (s != CUBLAS_STATUS_SUCCESS) {                                 \
            std::cerr << "cuBLAS error " << __FILE__ << ':' << __LINE__    \
                      << std::endl;                                      \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                 \
    } while (0)

// ---------------------------------------------------------------------
// Integer ceiling division (kept for consistency with the other files)
// ---------------------------------------------------------------------
#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

// ---------------------------------------------------------------------
// Kernel – 2‑D block tiling with vectorised loads/stores
// ---------------------------------------------------------------------
template <const int BM, const int BN, const int BK, const int TM, const int TN>
__global__ void __launch_bounds__((BM * BN) / (TM * TN), 1)
Vectorize(int M, int N, int K,
               float alpha, float *A,
               float *B, float beta, float *C)
{
    // -------------------------------------------------------------
    // 1) Identify the output tile this block will compute
    // -------------------------------------------------------------
    const uint cRow = blockIdx.y;                     // tile‑row index
    const uint cCol = blockIdx.x;                     // tile‑col index

    // BN/TN threads span a column of the tile
    const int threadCol = threadIdx.x % (BN / TN);
    const int threadRow = threadIdx.x / (BN / TN);

    // -------------------------------------------------------------
    // 2) Shared‑memory tiles for A and B
    // -------------------------------------------------------------
    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    // Move the global pointers to the first element of this tile
    A += cRow * BM * K;                     // start of A‑tile rows
    B += cCol * BN;                         // start of B‑tile columns
    C += cRow * BM * N + cCol * BN;         // start of C‑tile

    // -------------------------------------------------------------
    // 3) Indices used for loading the shared‑memory tiles
    //    (vectorised: 4 floats == one float4 per thread)
    // -------------------------------------------------------------
    const uint innerRowA = threadIdx.x / (BK / 4);
    const uint innerColA = threadIdx.x % (BK / 4);
    const uint innerRowB = threadIdx.x / (BN / 4);
    const uint innerColB = threadIdx.x % (BN / 4);

    // -------------------------------------------------------------
    // 4) Thread‑local accumulators (register‑blocked)
    // -------------------------------------------------------------
    float threadResults[TM * TN] = {0.0f};
    float regM[TM] = {0.0f};
    float regN[TN] = {0.0f};

    // -------------------------------------------------------------
    // 5) Main K‑loop – load tiles, compute, advance pointers
    // -------------------------------------------------------------
    for (uint bkIdx = 0; bkIdx < K; bkIdx += BK)
    {
        // ---- load a BM×BK tile of A (transpose while loading) ----
        float4 tmpA = reinterpret_cast<float4 *>(&A[innerRowA * K + innerColA * 4])[0];
        As[(innerColA * 4 + 0) * BM + innerRowA] = tmpA.x;
        As[(innerColA * 4 + 1) * BM + innerRowA] = tmpA.y;
        As[(innerColA * 4 + 2) * BM + innerRowA] = tmpA.z;
        As[(innerColA * 4 + 3) * BM + innerRowA] = tmpA.w;

        // ---- load a BK×BN tile of B (no transpose) ---------------
        reinterpret_cast<float4 *>(&Bs[innerRowB * BN + innerColB * 4])[0] =
            reinterpret_cast<float4 *>(&B[innerRowB * N + innerColB * 4])[0];

        __syncthreads();

        // ---- advance global pointers to the next K‑chunk ----------
        A += BK;          // move right by BK columns in A
        B += BK * N;      // move down by BK rows in B

        // ---- compute the partial products for this K‑chunk -------
        for (uint dotIdx = 0; dotIdx < BK; ++dotIdx)
        {
            // load TM elements of the current column of A into registers
            for (uint i = 0; i < TM; ++i)
                regM[i] = As[dotIdx * BM + threadRow * TM + i];

            // load TN elements of the current row of B into registers
            for (uint i = 0; i < TN; ++i)
                regN[i] = Bs[dotIdx * BN + threadCol * TN + i];

            // accumulate TM × TN products
            for (uint m = 0; m < TM; ++m)
                for (uint n = 0; n < TN; ++n)
                    threadResults[m * TN + n] += regM[m] * regN[n];
        }

        __syncthreads();   // ensure all threads finished before next tile
    }

    // -------------------------------------------------------------
    // 6) Write the TM × TN results back to global memory (vectorised)
    // -------------------------------------------------------------
    for (uint m = 0; m < TM; ++m)
    {
        for (uint n = 0; n < TN; n += 4)   // store 4 floats at a time
        {
            // load existing C values as a float4
            float4 tmpC = reinterpret_cast<float4 *>(
                &C[(threadRow * TM + m) * N + threadCol * TN + n])[0];

            // GEMM update
            tmpC.x = alpha * threadResults[m * TN + n]     + beta * tmpC.x;
            tmpC.y = alpha * threadResults[m * TN + n + 1] + beta * tmpC.y;
            tmpC.z = alpha * threadResults[m * TN + n + 2] + beta * tmpC.z;
            tmpC.w = alpha * threadResults[m * TN + n + 3] + beta * tmpC.w;

            // write back
            reinterpret_cast<float4 *>(
                &C[(threadRow * TM + m) * N + threadCol * TN + n])[0] = tmpC;
        }
    }
}

// ---------------------------------------------------------------------
// Launcher – mirrors the style of the other kernels
// ---------------------------------------------------------------------
void runVectorize(int M, int N, int K,
                       float alpha, float *A, float *B,
                       float beta, float *C)
{
    constexpr uint BK = 8;
    constexpr uint TM = 8;
    constexpr uint TN = 8;

    if (M >= 128 && N >= 128)
    {
        constexpr uint BM = 128;
        constexpr uint BN = 128;
        dim3 gridDim( CEIL_DIV(N, BN), CEIL_DIV(M, BM) );
        dim3 blockDim( (BM * BN) / (TM * TN) );   // (128*128)/(8*8) = 256 threads
        Vectorize<BM, BN, BK, TM, TN>
            <<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
    }
    else
    {
        // fallback for small problems (still a power‑of‑two tile size)
        constexpr uint BM = 64;
        constexpr uint BN = 64;
        dim3 gridDim( CEIL_DIV(N, BN), CEIL_DIV(M, BM) );
        dim3 blockDim( (BM * BN) / (TM * TN) );   // (64*64)/(8*8) = 64 threads
        Vectorize<BM, BN, BK, TM, TN>
            <<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
    }
}

// ---------------------------------------------------------------------
// Fill a buffer with deterministic pseudo‑random numbers [0,1)
// ---------------------------------------------------------------------
static void random_fill(float* p, size_t n)
{
    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    for (size_t i = 0; i < n; ++i) p[i] = dist(rng);
}

// ---------------------------------------------------------------------
// GFLOPS = 2·M·N·K / (seconds·1e9)
// ---------------------------------------------------------------------
static double gflops(int M, int N, int K, double sec)
{
    return 2.0 * M * N * K / (sec * 1e9);
}

// ---------------------------------------------------------------------
// Main driver – prints the exact format required by the assignment
// ---------------------------------------------------------------------
int main()
{
    std::cout << "Running Vectorized 2D‑Block‑Tiling Implementation" << std::endl;

    const int maxSize = 4096;
    const int sizes[] = {128, 256, 512, 1024, 2048, 4096};
    const int nSizes   = sizeof(sizes) / sizeof(sizes[0]);

    // --------------------------------------------------------------
    // Allocate host buffers for the *maximum* matrix size
    // --------------------------------------------------------------
    size_t maxElems = static_cast<size_t>(maxSize) * maxSize;
    float *hA = (float*)malloc(maxElems * sizeof(float));
    float *hB = (float*)malloc(maxElems * sizeof(float));
    float *hC = (float*)malloc(maxElems * sizeof(float));
    if (!hA || !hB || !hC) {
        std::cerr << "Host allocation failed\n";
        return EXIT_FAILURE;
    }

    random_fill(hA, maxElems);
    random_fill(hB, maxElems);
    random_fill(hC, maxElems);

    // --------------------------------------------------------------
    // Allocate device buffers (same maximum size) and copy once
    // --------------------------------------------------------------
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dB, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dC, maxElems * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(dA, hA, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dC, hC, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));

    // --------------------------------------------------------------
    // Benchmark parameters
    // --------------------------------------------------------------
    const float alpha = 0.5f;
    const float beta  = 3.0f;
    const int   repeat = 50;               // timed launches per size

    // --------------------------------------------------------------
    // Loop over the six matrix sizes
    // --------------------------------------------------------------
    for (int i = 0; i < nSizes; ++i) {
        int M = sizes[i];
        int N = sizes[i];
        int K = sizes[i];

        // Warm‑up launch (removes first‑run overhead)
        runVectorize(M, N, K, alpha, dA, dB, beta, dC);
        CUDA_CHECK(cudaDeviceSynchronize());

        // ----------------------------------------------------------
        // Timing with CUDA events
        // ----------------------------------------------------------
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaEventRecord(start));
        for (int r = 0; r < repeat; ++r) {
            runVectorize(M, N, K, alpha, dA, dB, beta, dC);
        }
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        ms /= repeat;                     // average per launch (ms)

        double sec = ms * 1e-3;           // seconds for printing
        double perf = gflops(M, N, K, sec);

        // ----- EXACT output format -----
        printf("dimensions(m=n=k) %d, alpha: %.1f, beta: %.1f\n",
               M, alpha, beta);
        printf("Average elapsed time: (%.6f) s, performance: (%8.1f) GFLOPS. size: (%d).\n",
               sec, perf, M);
        // ------------------------------

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }

    // --------------------------------------------------------------
    // Clean‑up
    // --------------------------------------------------------------
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA);
    free(hB);
    free(hC);

    return 0;
}

Writing vectorize.cu


In [18]:
!nvcc -arch=sm_75 -lcublas vectorize.cu -o vectorize.exe
!./vectorize.exe

Running Vectorized 2D‑Block‑Tiling Implementation
dimensions(m=n=k) 128, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000093) s, performance: (    44.9) GFLOPS. size: (128).
dimensions(m=n=k) 256, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000170) s, performance: (   197.1) GFLOPS. size: (256).
dimensions(m=n=k) 512, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000327) s, performance: (   820.6) GFLOPS. size: (512).
dimensions(m=n=k) 1024, alpha: 0.5, beta: 3.0
Average elapsed time: (0.001425) s, performance: (  1506.8) GFLOPS. size: (1024).
dimensions(m=n=k) 2048, alpha: 0.5, beta: 3.0
Average elapsed time: (0.005369) s, performance: (  3199.7) GFLOPS. size: (2048).
dimensions(m=n=k) 4096, alpha: 0.5, beta: 3.0
Average elapsed time: (0.037373) s, performance: (  3677.5) GFLOPS. size: (4096).


In [19]:
%%writefile autotuning.cu
//==========================================================================
// autotuning.cu – GEMM with an auto‑tuned 2‑D block‑tiling kernel Best Model So Far.
//==========================================================================

#ifndef uint
#define uint unsigned int
#endif

#include <iostream>
#include <random>
#include <algorithm>
#include <cstdio>
#include <cstdlib>
#include <cassert>

#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CEIL_DIV(M, N) (((M) + (N)-1) / (N))

// ---------------------------------------------------------------------
// Simple error‑checking macro
// ---------------------------------------------------------------------
#define CUDA_CHECK(err)                                                   \
    do {                                                                 \
        cudaError_t e = (err);                                           \
        if (e != cudaSuccess) {                                          \
            std::cerr << "CUDA error " << __FILE__ << ':' << __LINE__     \
                      << " : " << cudaGetErrorString(e) << std::endl;     \
            std::exit(EXIT_FAILURE);                                    \
        }                                                                \
    } while (0)

// ---------------------------------------------------------------------
// Kernel – auto‑tuned 2‑D block tiling
// ---------------------------------------------------------------------
constexpr int K9_NUM_THREADS = 256;

template <const int BM, const int BN, const int BK,
          const int TM, const int TN>
__global__ void __launch_bounds__(K9_NUM_THREADS, 1)
sgemmAutotuned(int M, int N, int K,
               float alpha, const float *A, const float *B,
               float beta,  float *C)
{
    const uint cRow = blockIdx.y;
    const uint cCol = blockIdx.x;

    // -----------------------------------------------------------------
    // Warptile dimensions (each warp = 16 threads)
    // -----------------------------------------------------------------
    constexpr int WM = TM * 16;
    constexpr int WN = TN * 16;
    constexpr int WMITER = CEIL_DIV(BM, WM);
    constexpr int WNITER = CEIL_DIV(BN, WN);

    // Thread position inside a warptile
    const int threadCol = threadIdx.x % (WN / TN);
    const int threadRow = threadIdx.x / (WN / TN);

    // Shared‑memory tiles
    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    // Point to the first tile of this block
    A += cRow * BM * K;
    B += cCol * BN;
    C += cRow * BM * N + cCol * BN;

    // -----------------------------------------------------------------
    // Indices for loading 4‑element vectors (float4) from global memory
    // -----------------------------------------------------------------
    const uint innerRowA = threadIdx.x / (BK / 4);
    const uint innerColA = threadIdx.x % (BK / 4);
    constexpr uint rowStrideA = (K9_NUM_THREADS * 4) / BK;

    const uint innerRowB = threadIdx.x / (BN / 4);
    const uint innerColB = threadIdx.x % (BN / 4);
    constexpr uint rowStrideB = K9_NUM_THREADS / (BN / 4);

    // -----------------------------------------------------------------
    // Per‑thread accumulator (register file)
    // -----------------------------------------------------------------
    float threadResults[WMITER * WNITER * TM * TN] = {0.0f};
    float regM[TM] = {0.0f};
    float regN[TN] = {0.0f};

    // -----------------------------------------------------------------
    // Main K‑loop – load tiles, compute, advance pointers
    // -----------------------------------------------------------------
    for (uint bkIdx = 0; bkIdx < K; bkIdx += BK) {
        // ---- load A tile (transpose while storing) --------------------
        for (uint offset = 0; offset + rowStrideA <= BM; offset += rowStrideA) {
            float4 tmp = reinterpret_cast<const float4 *>(
                &A[(innerRowA + offset) * K + innerColA * 4])[0];

            As[(innerColA * 4 + 0) * BM + innerRowA + offset] = tmp.x;
            As[(innerColA * 4 + 1) * BM + innerRowA + offset] = tmp.y;
            As[(innerColA * 4 + 2) * BM + innerRowA + offset] = tmp.z;
            As[(innerColA * 4 + 3) * BM + innerRowA + offset] = tmp.w;
        }

        // ---- load B tile ------------------------------------------------
        for (uint offset = 0; offset + rowStrideB <= BK; offset += rowStrideB) {
            reinterpret_cast<float4 *>(
                &Bs[(innerRowB + offset) * BN + innerColB * 4])[0] =
                reinterpret_cast<const float4 *>(
                    &B[(innerRowB + offset) * N + innerColB * 4])[0];
        }

        __syncthreads();

        // ---- compute ----------------------------------------------------
        for (uint wmIdx = 0; wmIdx < WMITER; ++wmIdx) {
            for (uint wnIdx = 0; wnIdx < WNITER; ++wnIdx) {
                for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
                    // load a column of A into registers
                    for (uint i = 0; i < TM; ++i)
                        regM[i] = As[dotIdx * BM +
                                   (wmIdx * WM) + threadRow * TM + i];

                    // load a row of B into registers
                    for (uint i = 0; i < TN; ++i)
                        regN[i] = Bs[dotIdx * BN +
                                   (wnIdx * WN) + threadCol * TN + i];

                    // accumulate
                    for (uint m = 0; m < TM; ++m)
                        for (uint n = 0; n < TN; ++n)
                            threadResults[(wmIdx * TM + m) *
                                          (WNITER * TN) +
                                          wnIdx * TN + n] +=
                                regM[m] * regN[n];
                }
            }
        }

        __syncthreads();

        // advance to the next K‑tile
        A += BK;          // right by BK columns
        B += BK * N;      // down by BK rows
    }

    // -----------------------------------------------------------------
    // Write results back to global memory (vectorized stores)
    // -----------------------------------------------------------------
    for (uint wmIdx = 0; wmIdx < WMITER; ++wmIdx) {
        for (uint wnIdx = 0; wnIdx < WNITER; ++wnIdx) {
            float *C_interim = C + (wmIdx * WM * N) + (wnIdx * WN);

            for (uint m = 0; m < TM; ++m) {
                for (uint n = 0; n < TN; n += 4) {
                    // load 4‑element vector of C
                    float4 tmp = reinterpret_cast<float4 *>(
                        &C_interim[(threadRow * TM + m) * N +
                                   threadCol * TN + n])[0];

                    // update with GEMM result
                    const int idxBase = (wmIdx * TM + m) *
                                       (WNITER * TN) + wnIdx * TN + n;
                    tmp.x = alpha * threadResults[idxBase + 0] + beta * tmp.x;
                    tmp.y = alpha * threadResults[idxBase + 1] + beta * tmp.y;
                    tmp.z = alpha * threadResults[idxBase + 2] + beta * tmp.z;
                    tmp.w = alpha * threadResults[idxBase + 3] + beta * tmp.w;

                    // store back
                    reinterpret_cast<float4 *>(
                        &C_interim[(threadRow * TM + m) * N +
                                   threadCol * TN + n])[0] = tmp;
                }
            }
        }
    }
}

// ---------------------------------------------------------------------
// Launcher – selects the tuned parameters for the target GPU
// ---------------------------------------------------------------------
void runSgemmAutotuned(int M, int N, int K,
                       float alpha, float *A, float *B,
                       float beta,  float *C)
{
    // Parameters tuned for an RTX A6000 (FP32)
    const uint K9_BK = 16;
    const uint K9_TM = 8;
    const uint K9_TN = 8;
    const uint K9_BM = 128;
    const uint K9_BN = 128;

    dim3 blockDim(K9_NUM_THREADS);
    dim3 gridDim(CEIL_DIV(N, K9_BN), CEIL_DIV(M, K9_BM));

    // sanity checks (identical to the original source)
    static_assert((K9_NUM_THREADS * 4) % K9_BK == 0,
                  "NUM_THREADS*4 must be multiple of K9_BK");
    static_assert((K9_NUM_THREADS * 4) % K9_BN == 0,
                  "NUM_THREADS*4 must be multiple of K9_BN");
    static_assert(K9_BN % (16 * K9_TN) == 0,
                  "K9_BN must be a multiple of 16*K9_TN");
    static_assert(K9_BM % (16 * K9_TM) == 0,
                  "K9_BM must be a multiple of 16*K9_TM");
    static_assert((K9_BM * K9_BK) % (4 * K9_NUM_THREADS) == 0,
                  "K9_BM*K9_BK must be a multiple of 4*NUM_THREADS");
    static_assert((K9_BN * K9_BK) % (4 * K9_NUM_THREADS) == 0,
                  "K9_BN*K9_BK must be a multiple of 4*NUM_THREADS");

    sgemmAutotuned<K9_BM, K9_BN, K9_BK, K9_TM, K9_TN>
        <<<gridDim, blockDim>>>(M, N, K, alpha, A, B, beta, C);
}

// ---------------------------------------------------------------------
// Helper utilities (same as the other samples)
// ---------------------------------------------------------------------
static void random_fill(float *p, size_t n)
{
    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    for (size_t i = 0; i < n; ++i) p[i] = dist(rng);
}

static double gflops(int M, int N, int K, double sec)
{
    return 2.0 * M * N * K / (sec * 1e9);
}

// ---------------------------------------------------------------------
// Main driver – identical benchmarking harness to the other examples
// ---------------------------------------------------------------------
int main()
{
    std::cout << "Running Auto‑tuned GEMM (2D block tiling)" << std::endl;

    const int maxSize = 4096;
    const int sizes[] = {128, 256, 512, 1024, 2048, 4096};
    const int nSizes = sizeof(sizes) / sizeof(sizes[0]);

    size_t maxElems = static_cast<size_t>(maxSize) * maxSize;
    float *hA = (float*)malloc(maxElems * sizeof(float));
    float *hB = (float*)malloc(maxElems * sizeof(float));
    float *hC = (float*)malloc(maxElems * sizeof(float));
    if (!hA || !hB || !hC) {
        std::cerr << "Host allocation failed\n";
        return EXIT_FAILURE;
    }

    random_fill(hA, maxElems);
    random_fill(hB, maxElems);
    random_fill(hC, maxElems);

    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dB, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dC, maxElems * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(dA, hA, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dC, hC, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));

    const float alpha = 0.5f;
    const float beta  = 3.0f;
    const int   repeat = 50;

    for (int i = 0; i < nSizes; ++i) {
        int M = sizes[i];
        int N = sizes[i];
        int K = sizes[i];

        // warm‑up
        runSgemmAutotuned(M, N, K, alpha, dA, dB, beta, dC);
        CUDA_CHECK(cudaDeviceSynchronize());

        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaEventRecord(start));
        for (int r = 0; r < repeat; ++r) {
            runSgemmAutotuned(M, N, K, alpha, dA, dB, beta, dC);
        }
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        ms /= repeat;

        double sec = ms * 1e-3;
        double perf = gflops(M, N, K, sec);

        printf("dimensions(m=n=k) %d, alpha: %.1f, beta: %.1f\n",
               M, alpha, beta);
        printf("Average elapsed time: (%.6f) s, performance: (%8.1f) GFLOPS. size: (%d).\n",
               sec, perf, M);

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }

    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA);
    free(hB);
    free(hC);
    return 0;
}

Writing autotuning.cu


In [20]:
!nvcc -arch=sm_75 -lcublas autotuning.cu -o autotuning.exe
!./autotuning.exe

Running Auto‑tuned GEMM (2D block tiling)
dimensions(m=n=k) 128, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000085) s, performance: (    49.2) GFLOPS. size: (128).
dimensions(m=n=k) 256, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000156) s, performance: (   215.5) GFLOPS. size: (256).
dimensions(m=n=k) 512, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000299) s, performance: (   899.1) GFLOPS. size: (512).
dimensions(m=n=k) 1024, alpha: 0.5, beta: 3.0
Average elapsed time: (0.001400) s, performance: (  1534.3) GFLOPS. size: (1024).
dimensions(m=n=k) 2048, alpha: 0.5, beta: 3.0
Average elapsed time: (0.005609) s, performance: (  3062.6) GFLOPS. size: (2048).
dimensions(m=n=k) 4096, alpha: 0.5, beta: 3.0
Average elapsed time: (0.037280) s, performance: (  3686.7) GFLOPS. size: (4096).


In [21]:
%%writefile cublas_bench.cu
#ifndef uint
#define uint unsigned int
#endif

#include <iostream>
#include <random>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <cstdint>          // for uint32_t
#include <cassert>

// ---------------------------------------------------------------------
// Simple error‑checking macros (CUDA and cuBLAS)
// ---------------------------------------------------------------------
#define CUDA_CHECK(call)                                                   \
    do {                                                                   \
        cudaError_t err = (call);                                          \
        if (err != cudaSuccess) {                                          \
            std::cerr << "CUDA error " << __FILE__ << ':' << __LINE__       \
                      << " : " << cudaGetErrorString(err) << std::endl;     \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                  \
    } while (0)

#define CUBLAS_CHECK(call)                                                 \
    do {                                                                   \
        cublasStatus_t stat = (call);                                      \
        if (stat != CUBLAS_STATUS_SUCCESS) {                               \
            std::cerr << "cuBLAS error " << __FILE__ << ':' << __LINE__     \
                      << std::endl;                                        \
            std::exit(EXIT_FAILURE);                                      \
        }                                                                  \
    } while (0)

// ---------------------------------------------------------------------
// Integer ceiling division (kept for consistency with other files)
// ---------------------------------------------------------------------
#define CEIL_DIV(x, y)   ( ((x) + (y) - 1) / (y) )

// ---------------------------------------------------------------------
// Fill a buffer with deterministic pseudo‑random numbers [0,1)
// Column‑major layout (element (r,c) is at p[c*rows + r])
// ---------------------------------------------------------------------
static void random_fill_colmajor(float *p, size_t rows, size_t cols)
{
    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(0.0f, 1.0f);
    for (size_t c = 0; c < cols; ++c)
        for (size_t r = 0; r < rows; ++r)
            p[c * rows + r] = dist(rng);
}

// ---------------------------------------------------------------------
// Compute GFLOPS for a single SGEMM of size M×N×K
// ---------------------------------------------------------------------
static double gflops(int M, int N, int K, double seconds)
{
    double flops = 2.0 * static_cast<double>(M) *
                         static_cast<double>(N) *
                         static_cast<double>(K);
    return (flops * 1e-9) / seconds;   // GFLOP/s
}

// ---------------------------------------------------------------------
// Main driver
// ---------------------------------------------------------------------
int main()
{
    std::cout << "Running cuBLAS SGEMM Implementation" << std::endl;

    const float alpha = 0.5f;
    const float beta  = 3.0f;
    const int   repeat = 50;               // timed launches per size
    const int   warm_up = 5;               // warm‑up launches

    // -----------------------------------------------------------------
    // Square matrix sizes to benchmark
    // -----------------------------------------------------------------
    const int sizes[] = {128, 256, 512, 1024, 2048, 4096};
    const int nSizes   = sizeof(sizes) / sizeof(sizes[0]);

    // -----------------------------------------------------------------
    // Allocate host buffers for the *maximum* matrix size (4096×4096)
    // -----------------------------------------------------------------
    const int maxSize = 4096;
    const size_t maxElems = static_cast<size_t>(maxSize) * maxSize;

    float *hA = (float*)malloc(maxElems * sizeof(float));
    float *hB = (float*)malloc(maxElems * sizeof(float));
    float *hC = (float*)malloc(maxElems * sizeof(float));
    if (!hA || !hB || !hC) {
        std::cerr << "Host allocation failed\n";
        return EXIT_FAILURE;
    }

    // Initialise with deterministic data (column‑major)
    random_fill_colmajor(hA, maxSize, maxSize);
    random_fill_colmajor(hB, maxSize, maxSize);
    // hC will be overwritten by cuBLAS, no need to initialise

    // -----------------------------------------------------------------
    // Allocate device buffers (same maximum size)
    // -----------------------------------------------------------------
    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dB, maxElems * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dC, maxElems * sizeof(float)));

    // Transfer the *largest* matrices once; smaller sizes will just use a
    // prefix of these buffers.
    CUDA_CHECK(cudaMemcpy(dA, hA, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB, maxElems * sizeof(float),
                          cudaMemcpyHostToDevice));

    // -----------------------------------------------------------------
    // Create cuBLAS handle (single handle reused for all sizes)
    // -----------------------------------------------------------------
    cublasHandle_t handle;
    CUBLAS_CHECK(cublasCreate(&handle));

    // -----------------------------------------------------------------
    // Loop over the requested matrix sizes
    // -----------------------------------------------------------------
    for (int i = 0; i < nSizes; ++i) {
        const int M = sizes[i];
        const int N = sizes[i];
        const int K = sizes[i];

        // -------------------------------------------------------------
        // Warm‑up launches (remove first‑run overhead)
        // -------------------------------------------------------------
        for (int w = 0; w < warm_up; ++w) {
            CUBLAS_CHECK(cublasSgemm(handle,
                                     CUBLAS_OP_N, CUBLAS_OP_N,
                                     N, M, K,               // note: cuBLAS is column‑major
                                     &alpha,
                                     dB, N,                 // B is N×K
                                     dA, K,                 // A is K×M
                                     &beta,
                                     dC, N));                // C is N×M
        }
        CUDA_CHECK(cudaDeviceSynchronize());

        // -------------------------------------------------------------
        // Timing with CUDA events
        // -------------------------------------------------------------
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        CUDA_CHECK(cudaEventRecord(start));
        for (int r = 0; r < repeat; ++r) {
            CUBLAS_CHECK(cublasSgemm(handle,
                                     CUBLAS_OP_N, CUBLAS_OP_N,
                                     N, M, K,
                                     &alpha,
                                     dB, N,
                                     dA, K,
                                     &beta,
                                     dC, N));
        }
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));

        float msTotal = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&msTotal, start, stop));
        double secPer = static_cast<double>(msTotal) / 1000.0 / repeat; // seconds

        double perf = gflops(M, N, K, secPer);

        // ----- EXACT output format -----
        printf("dimensions(m=n=k) %d, alpha: %.1f, beta: %.1f\n",
               M, alpha, beta);
        printf("Average elapsed time: (%.6f) s, performance: (%8.1f) GFLOPS. size: (%d).\n",
               secPer, perf, M);
        // ------------------------------

        CUDA_CHECK(cudaEventDestroy(start));
        CUDA_CHECK(cudaEventDestroy(stop));
    }

    // -----------------------------------------------------------------
    // Clean‑up
    // -----------------------------------------------------------------
    CUBLAS_CHECK(cublasDestroy(handle));
    CUDA_CHECK(cudaFree(dA));
    CUDA_CHECK(cudaFree(dB));
    CUDA_CHECK(cudaFree(dC));
    free(hA);
    free(hB);
    free(hC);

    return 0;
}

Writing cublas_bench.cu


In [22]:
!nvcc -arch=sm_75 -lcublas cublas_bench.cu -o cublas_bench.exe
!./cublas_bench.exe

Running cuBLAS SGEMM Implementation
dimensions(m=n=k) 128, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000015) s, performance: (   275.1) GFLOPS. size: (128).
dimensions(m=n=k) 256, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000049) s, performance: (   681.9) GFLOPS. size: (256).
dimensions(m=n=k) 512, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000133) s, performance: (  2011.8) GFLOPS. size: (512).
dimensions(m=n=k) 1024, alpha: 0.5, beta: 3.0
Average elapsed time: (0.000829) s, performance: (  2592.0) GFLOPS. size: (1024).
dimensions(m=n=k) 2048, alpha: 0.5, beta: 3.0
Average elapsed time: (0.004841) s, performance: (  3548.9) GFLOPS. size: (2048).
dimensions(m=n=k) 4096, alpha: 0.5, beta: 3.0
Average elapsed time: (0.031338) s, performance: (  4385.6) GFLOPS. size: (4096).
